# Data Preparation and Integration

## Project: E-commerce Sales Analysis (2024–2025)

### Objective
The objective of this stage is to prepare the cleaned datasets for analysis and integrate them into a unified analytical dataset.

### Tasks
1. Load the cleaned datasets.
2. Validate primary keys and foreign keys.
3. Identify unmatched records between datasets.
4. Integrate datasets using appropriate joins.
5. Validate the integrated dataset.
6. Save the final prepared datasets for Exploratory Data Analysis (EDA).

### Source Datasets
- Orders
- Order Items
- Products
- Customers

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt
import openpyxl as xl

In [4]:
# Load cleaned datasets

orders = pd.read_csv(
    "../data/cleaned/orders_cleaned.csv",
    parse_dates=["order_date"]
)

order_items = pd.read_csv(
    "../data/cleaned/order_items_cleaned.csv"
)

products = pd.read_csv(
    "../data/cleaned/products_cleaned.csv"
)

customers = pd.read_csv(
    "../data/cleaned/customers_cleaned.csv",
    parse_dates=["signup_date"]
)

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [5]:
#checking dataset
datasets = {
    "Orders": orders,
    "Order Items": order_items,
    "Products": products,
    "Customers": customers
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())


Orders
Shape: (9207, 7)
Columns: ['order_id', 'customer_id', 'order_date', 'status', 'payment_method', 'discount_code', 'shipping_fee']

Order Items
Shape: (15608, 6)
Columns: ['order_id', 'product_id', 'quantity', 'unit_price', 'discount_amount', 'price_was_missing']

Products
Shape: (90, 5)
Columns: ['product_id', 'product_name', 'category', 'list_price', 'unit_cost']

Customers
Shape: (3600, 4)
Columns: ['customer_id', 'signup_date', 'state', 'acquisition_channel']


## Primary Key Validation

### Objective
Verify that the primary key columns uniquely identify records in their respective datasets.

### Expected Primary Keys
- Orders: `order_id`
- Products: `product_id`
- Customers: `customer_id`

A valid primary key should not contain duplicate or missing values.



### Primary Key Validation – Results

The primary key columns of the Orders, Products, and Customers datasets were examined for missing and duplicate values.

**Findings:**

| Dataset | Primary Key | Missing | Duplicates |
|---|---|---:|---:|
| Orders | order_id | 0 | 0 |
| Products | product_id | 0 | 0 |
| Customers | customer_id | 0 | 0 |

**Conclusion:**

All three datasets passed primary key validation. Each primary key uniquely identifies its records, and no missing or duplicate primary key values were found.

No corrections were required.

In [6]:
# Check primary key uniqueness

primary_keys = {
    "Orders": (orders, "order_id"),
    "Products": (products, "product_id"),
    "Customers": (customers, "customer_id")
}

for name, (df, key) in primary_keys.items():

    print(f"\n{name}")
    print("Missing keys:", df[key].isnull().sum())
    print("Duplicate keys:", df[key].duplicated().sum())
    print("Unique keys:", df[key].nunique())


Orders
Missing keys: 0
Duplicate keys: 0
Unique keys: 9207

Products
Missing keys: 0
Duplicate keys: 0
Unique keys: 90

Customers
Missing keys: 0
Duplicate keys: 0
Unique keys: 3600


### Foreign Key Validation

**Objective:**

Verify that foreign key values in the Orders and Order Items datasets have corresponding records in their referenced datasets.

**Expected relationships:**

- `orders.customer_id` → `customers.customer_id`
- `order_items.order_id` → `orders.order_id`
- `order_items.product_id` → `products.product_id`

This validation helps identify orphan records that cannot be linked to their corresponding customers, orders, or products.


### Foreign Key Validation – Results

Foreign key relationships were checked across the four datasets to identify unmatched records.

**Relationships examined:**

| Child Dataset | Foreign Key | Parent Dataset | Unmatched Records |
|---|---|---|---:|
| Orders | customer_id | Customers | 0 |
| Order Items | order_id | Orders | 0 |
| Order Items | product_id | Products | 0 |

**Findings:**

- All orders reference existing customers.
- All order-item records reference existing orders.
- All order-item records reference existing products.
- No orphan records or unmatched foreign keys were identified.

**Conclusion:**

All foreign key relationships passed validation. The four datasets are structurally connected and ready for integration.


In [8]:
# Check foreign key relationships

foreign_key_checks = {
    "Orders -> Customers": (
        orders, "customer_id",
        customers, "customer_id"
    ),
    "Order Items -> Orders": (
        order_items, "order_id",
        orders, "order_id"
    ),
    "Order Items -> Products": (
        order_items, "product_id",
        products, "product_id"
    )
}

for name, (child_df, child_key, parent_df, parent_key) in foreign_key_checks.items():

    unmatched = ~child_df[child_key].isin(parent_df[parent_key])

    print(f"\n{name}")
    print("Unmatched records:", unmatched.sum())


Orders -> Customers
Unmatched records: 0

Order Items -> Orders
Unmatched records: 0

Order Items -> Products
Unmatched records: 0


### Step 1: Merge Order Items with Orders

We are merging the `order_items` dataset with the `orders` dataset using `order_id`.

The `order_items` dataset is our left/base table because each row represents an individual product within an order.

We use a left join to retain all order-item records. We also use `validate="many_to_one"` to ensure that multiple order-item rows can match one order, but each order must appear only once in the orders dataset.

### Step 1: Merge Validation Results

The `order_items` dataset was successfully merged with the `orders` dataset using a left join on `order_id`.

**Validation results:**
- Original order-item records: 15,608
- Merged records: 15,608
- Matched records: 15,608
- Unmatched records: 0
- Missing customer IDs, order dates, and statuses: 0

The `many_to_one` relationship validation passed.

**Conclusion:** All order-item records were successfully matched with their corresponding order information. No records were lost or duplicated during the merge.

**Note:** Shipping fees are stored at the order level and may repeat across multiple order-item rows. They must be aggregated separately when calculating shipping-related metrics.

In [9]:
order_items_orders = order_items.merge(
    orders,
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

In [10]:
#validating the merge
print("Merged dataset shape:", order_items_orders.shape)

print("\nMerge validation:")
print(order_items_orders["_merge"].value_counts())

print("\nMissing values in order information:")
print(
    order_items_orders[
        ["customer_id", "order_date", "status"]
    ].isnull().sum()
)

Merged dataset shape: (15608, 13)

Merge validation:
_merge
both          15608
left_only         0
right_only        0
Name: count, dtype: int64

Missing values in order information:
customer_id    0
order_date     0
status         0
dtype: int64


In [11]:
#removing the temprory validation colum
order_items_orders.drop(columns="_merge", inplace=True)

print("Updated shape:", order_items_orders.shape)

Updated shape: (15608, 12)


In [12]:
#inspecting the merge dataset
display(order_items_orders.head())

print("\nDataset information:")
order_items_orders.info()

,order_id,product_id,quantity,unit_price,discount_amount,price_was_missing,customer_id,order_date,status,payment_method,discount_code,shipping_fee
0,100001,1031,1,38.99,0.0,False,1530,2024-01-01 09:24:45,completed,credit_card,NaN,6.95
1,100002,1087,1,45.99,0.0,False,3531,2024-01-01 10:37:49,completed,credit_card,NaN,0.00
2,100002,1072,1,141.99,0.0,False,3531,2024-01-01 10:37:49,completed,credit_card,NaN,0.00
3,100003,1048,1,139.99,0.0,False,840,2024-01-01 10:59:09,completed,credit_card,NaN,0.00
4,100004,1083,1,19.99,2.0,False,725,2024-01-01 12:48:34,completed,apple_pay,WELCOME10,0.00



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15608 entries, 0 to 15607
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   order_id           15608 non-null  int64         
 1   product_id         15608 non-null  int64         
 2   quantity           15608 non-null  int64         
 3   unit_price         15608 non-null  float64       
 4   discount_amount    15608 non-null  float64       
 5   price_was_missing  15608 non-null  bool          
 6   customer_id        15608 non-null  int64         
 7   order_date         15608 non-null  datetime64[ns]
 8   status             15608 non-null  object        
 9   payment_method     15608 non-null  object        
 10  discount_code      2378 non-null   object        
 11  shipping_fee       15608 non-null  float64       
dtypes: bool(1), datetime64[ns](1), float64(3), int64(4), object(3)
memory usage: 1.3+ MB


### Step 2: Merge Validation Results

The order-item and orders dataset was successfully merged with the products dataset using a left join on `product_id`.

**Validation results:**
- Merged records: 15,608
- Matched product records: 15,608
- Unmatched product records: 0
- Missing product names: 0
- Missing categories: 0
- Missing unit costs: 0

The `many_to_one` relationship validation passed.

**Conclusion:** All order-item records were successfully matched with their corresponding product information. The merge did not result in any lost or duplicated rows.

**Additional note:** Historical transaction prices (`unit_price`) will be used for revenue calculations rather than current catalogue prices (`list_price`).

In [13]:
#merge with product
order_items_orders_products = order_items_orders.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

In [14]:
#validate the merge
print("Merged dataset shape:", order_items_orders_products.shape)

print("\nMerge validation:")
print(order_items_orders_products["_merge"].value_counts())

print("\nMissing product information:")
print(
    order_items_orders_products[
        ["product_name", "category", "unit_cost"]
    ].isnull().sum()
)

Merged dataset shape: (15608, 17)

Merge validation:
_merge
both          15608
left_only         0
right_only        0
Name: count, dtype: int64

Missing product information:
product_name    0
category        0
unit_cost       0
dtype: int64


In [16]:
#Remove the temporary validation column
order_items_orders_products.drop(
    columns="_merge",
    inplace=True
)

print("Updated shape:", order_items_orders_products.shape)

Updated shape: (15608, 16)


In [17]:
#Inspect the merged dataset
display(order_items_orders_products.head())

print("\nDataset information:")
order_items_orders_products.info()

,order_id,product_id,quantity,unit_price,discount_amount,price_was_missing,customer_id,order_date,status,payment_method,discount_code,shipping_fee,product_name,category,list_price,unit_cost
0,100001,1031,1,38.99,0.0,False,1530,2024-01-01 09:24:45,completed,credit_card,NaN,6.95,Stoneware Soap Dispenser,Bath,38.99,20.19
1,100002,1087,1,45.99,0.0,False,3531,2024-01-01 10:37:49,completed,credit_card,NaN,0.00,Cotton Pantry Container Set,Storage,45.99,20.30
2,100002,1072,1,141.99,0.0,False,3531,2024-01-01 10:37:49,completed,credit_card,NaN,0.00,Coastal Picture Frame Set,Decor,141.99,62.78
3,100003,1048,1,139.99,0.0,False,840,2024-01-01 10:59:09,completed,credit_card,NaN,0.00,Linen Loveseat,Furniture,139.99,71.41
4,100004,1083,1,19.99,2.0,False,725,2024-01-01 12:48:34,completed,apple_pay,WELCOME10,0.00,Velvet Drawer Organizer,Storage,19.99,9.51



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15608 entries, 0 to 15607
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   order_id           15608 non-null  int64         
 1   product_id         15608 non-null  int64         
 2   quantity           15608 non-null  int64         
 3   unit_price         15608 non-null  float64       
 4   discount_amount    15608 non-null  float64       
 5   price_was_missing  15608 non-null  bool          
 6   customer_id        15608 non-null  int64         
 7   order_date         15608 non-null  datetime64[ns]
 8   status             15608 non-null  object        
 9   payment_method     15608 non-null  object        
 10  discount_code      2378 non-null   object        
 11  shipping_fee       15608 non-null  float64       
 12  product_name       15608 non-null  object        
 13  category           15608 non-null  obje

### Step 3: Merge Validation Results

The integrated order-item, order, and product dataset was successfully merged with the customers dataset using a left join on `customer_id`.

**Validation results:**
- Final merged records: 15,608
- Matched customer records: 15,608
- Unmatched records: 0
- Missing signup dates: 0
- Missing states: 0
- Missing acquisition channels: 0

The `many_to_one` relationship validation passed.

**Conclusion:** All order-item records were successfully matched with their corresponding customer information. No records were lost or duplicated during the merge.

## Final Dataset Integration Summary

The four cleaned datasets — Orders, Order Items, Products, and Customers — were successfully integrated into one product-level analytical dataset.

**Integration approach:**
- Order Items + Orders using `order_id`
- Result + Products using `product_id`
- Result + Customers using `customer_id`

All merges used left joins with `validate="many_to_one"` to protect the expected table relationships.

**Final dataset:**
- Rows: 15,608
- Columns: 19
- Unmatched records across all merges: 0
- Missing product information: 0
- Missing customer information: 0

**Conclusion:** All order-item records were successfully integrated with their corresponding order, product, and customer information. The original row count was preserved throughout the integration process.

In [18]:
#merge with customer
final_dataset = order_items_orders_products.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

In [19]:
#validating the final merge
print("Final dataset shape:", final_dataset.shape)

print("\nMerge validation:")
print(final_dataset["_merge"].value_counts())

print("\nMissing customer information:")
print(
    final_dataset[
        ["signup_date", "state", "acquisition_channel"]
    ].isnull().sum()
)

Final dataset shape: (15608, 20)

Merge validation:
_merge
both          15608
left_only         0
right_only        0
Name: count, dtype: int64

Missing customer information:
signup_date            0
state                  0
acquisition_channel    0
dtype: int64


In [20]:
#removing the temprory validation column
final_dataset.drop(
    columns="_merge",
    inplace=True
)

print("Final dataset shape:", final_dataset.shape)

Final dataset shape: (15608, 19)


In [21]:
#Inspect the final integrated dataset
display(final_dataset.head())

print("\nDataset information:")
final_dataset.info()

,order_id,product_id,quantity,unit_price,discount_amount,price_was_missing,customer_id,order_date,status,payment_method,discount_code,shipping_fee,product_name,category,list_price,unit_cost,signup_date,state,acquisition_channel
0,100001,1031,1,38.99,0.0,False,1530,2024-01-01 09:24:45,completed,credit_card,NaN,6.95,Stoneware Soap Dispenser,Bath,38.99,20.19,2023-12-30,AZ,organic_search
1,100002,1087,1,45.99,0.0,False,3531,2024-01-01 10:37:49,completed,credit_card,NaN,0.00,Cotton Pantry Container Set,Storage,45.99,20.30,2023-11-19,VA,organic_search
2,100002,1072,1,141.99,0.0,False,3531,2024-01-01 10:37:49,completed,credit_card,NaN,0.00,Coastal Picture Frame Set,Decor,141.99,62.78,2023-11-19,VA,organic_search
3,100003,1048,1,139.99,0.0,False,840,2024-01-01 10:59:09,completed,credit_card,NaN,0.00,Linen Loveseat,Furniture,139.99,71.41,2023-10-30,MN,organic_search
4,100004,1083,1,19.99,2.0,False,725,2024-01-01 12:48:34,completed,apple_pay,WELCOME10,0.00,Velvet Drawer Organizer,Storage,19.99,9.51,2023-08-20,MI,paid_search



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15608 entries, 0 to 15607
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             15608 non-null  int64         
 1   product_id           15608 non-null  int64         
 2   quantity             15608 non-null  int64         
 3   unit_price           15608 non-null  float64       
 4   discount_amount      15608 non-null  float64       
 5   price_was_missing    15608 non-null  bool          
 6   customer_id          15608 non-null  int64         
 7   order_date           15608 non-null  datetime64[ns]
 8   status               15608 non-null  object        
 9   payment_method       15608 non-null  object        
 10  discount_code        2378 non-null   object        
 11  shipping_fee         15608 non-null  float64       
 12  product_name         15608 non-null  object        
 13  category 

In [22]:
# Check duplicate order-item combinations
print(
    "Duplicate order-item records:",
    final_dataset.duplicated(
        subset=["order_id", "product_id"]
    ).sum()
)

# Check missing values
print("\nMissing values:")
print(final_dataset.isnull().sum())

# Check order date range
print("\nOrder date range:")
print(final_dataset["order_date"].min())
print(final_dataset["order_date"].max())

# Check order statuses
print("\nOrder status distribution:")
print(final_dataset["status"].value_counts())

Duplicate order-item records: 0

Missing values:
order_id                   0
product_id                 0
quantity                   0
unit_price                 0
discount_amount            0
price_was_missing          0
customer_id                0
order_date                 0
status                     0
payment_method             0
discount_code          13230
shipping_fee               0
product_name               0
category                   0
list_price                 0
unit_cost                  0
signup_date                0
state                      0
acquisition_channel        0
dtype: int64

Order date range:
2024-01-01 09:24:45
2025-12-31 23:23:29

Order status distribution:
status
completed    14283
returned       766
canceled       559
Name: count, dtype: int64


## Creating Analytical Columns

After completing data integration and quality checks, we will create additional columns to support sales analysis.

**Date-related columns:**
- `year`: Extracts the year from the order date.
- `month`: Extracts the numerical month.
- `year_month`: Combines year and month for chronological monthly analysis.

**Sales-related columns:**
- `gross_amount`: Quantity multiplied by historical unit price.
- `net_amount`: Gross merchandise amount minus the line-level discount.

These calculations will be performed at the order-item level. All order statuses will be retained; revenue calculations will later filter for completed orders.

Shipping fees will remain separate because they are recorded at the order level.

In [23]:
# Extract date information
final_dataset["year"] = final_dataset["order_date"].dt.year

final_dataset["month"] = final_dataset["order_date"].dt.month

final_dataset["year_month"] = (
    final_dataset["order_date"].dt.to_period("M").astype(str)
)

# Calculate merchandise amounts
final_dataset["gross_amount"] = (
    final_dataset["quantity"] * final_dataset["unit_price"]
)

final_dataset["net_amount"] = (
    final_dataset["gross_amount"] - final_dataset["discount_amount"]
)

In [24]:
print("Updated dataset shape:", final_dataset.shape)

print("\nNew analytical columns:")
display(
    final_dataset[
        [
            "order_date",
            "year",
            "month",
            "year_month",
            "quantity",
            "unit_price",
            "gross_amount",
            "discount_amount",
            "net_amount"
        ]
    ].head()
)

print("\nMissing values in new columns:")
print(
    final_dataset[
        ["year", "month", "year_month", "gross_amount", "net_amount"]
    ].isnull().sum()
)

print("\nNegative net amounts:", (final_dataset["net_amount"] < 0).sum())

Updated dataset shape: (15608, 24)

New analytical columns:


,order_date,year,month,year_month,quantity,unit_price,gross_amount,discount_amount,net_amount
0,2024-01-01 09:24:45,2024,1,2024-01,1,38.99,38.99,0.0,38.99
1,2024-01-01 10:37:49,2024,1,2024-01,1,45.99,45.99,0.0,45.99
2,2024-01-01 10:37:49,2024,1,2024-01,1,141.99,141.99,0.0,141.99
3,2024-01-01 10:59:09,2024,1,2024-01,1,139.99,139.99,0.0,139.99
4,2024-01-01 12:48:34,2024,1,2024-01,1,19.99,19.99,2.0,17.99



Missing values in new columns:
year            0
month           0
year_month      0
gross_amount    0
net_amount      0
dtype: int64

Negative net amounts: 0


### Analytical Column Creation and Validation

Five analytical columns were successfully created:

- `year`: Extracted from the order date.
- `month`: Extracted from the order date.
- `year_month`: Created for monthly trend analysis.
- `gross_amount`: Calculated as quantity multiplied by historical unit price.
- `net_amount`: Calculated as gross amount minus the line-level discount.

**Validation results:**
- Final dataset: 15,608 rows and 24 columns.
- Missing values in the new columns: 0.
- Negative net amounts: 0.

**Conclusion:** All analytical columns were created successfully and passed the initial validation checks. The dataset is ready for saving and subsequent business analysis.

In [25]:


# Save final integrated dataset
final_dataset.to_csv(
    "../data/processed/ecommerce_integrated.csv",
    index=False
)

print("Final integrated dataset saved successfully!")

Final integrated dataset saved successfully!


In [26]:
# Load saved dataset
saved_dataset = pd.read_csv(
    "../data/processed/ecommerce_integrated.csv",
    parse_dates=["order_date", "signup_date"]
)

print("Saved dataset shape:", saved_dataset.shape)

print("Original dataset shape:", final_dataset.shape)

print(
    "Row count preserved:",
    len(saved_dataset) == len(final_dataset)
)

Saved dataset shape: (15608, 24)
Original dataset shape: (15608, 24)
Row count preserved: True
